# TrafficSense — Entrenamiento del Detector Especializado de Cascos (YOLO11)

**Proyecto:** TrafficSense — Monitoreo de flujo vehicular y detección de motociclistas sin casco  
**Institución:** Universidad Nacional de San Martín (UNSM)  
**Curso:** Proyectos de Inteligencia Artificial  
**Entorno recomendado:** Google Colab con acelerador GPU T4 (16 GB VRAM) o Kaggle Notebooks

---

## 1. Verificación de GPU y Entorno

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo: {torch.cuda.get_device_name(0)}")

## 2. Instalación de Ultralytics

In [ ]:
!pip install -q ultralytics supervision

## 3. Preparación del Dataset de Cascos
Descarga de datasets públicos (Kaggle Helmet Detection / Roboflow Universe) con partición 70/20/10.

In [ ]:
# Definición del archivo de configuración del dataset YAML
dataset_yaml = """
path: /content/dataset_helmet
train: images/train
val: images/val
test: images/test

names:
  0: helmet
  1: no_helmet
"""

with open("dataset_helmet.yaml", "w") as f:
    f.write(dataset_yaml.strip())

print("[✓] dataset_helmet.yaml creado exitosamente.")

## 4. Entrenamiento del Modelo (YOLO11 Nano / Small)
Se utiliza `yolo11n.pt` para optimizar latencia en CPU o `yolo11s.pt` si se busca mayor precisión.

In [ ]:
from ultralytics import YOLO

# Cargar pesos preentrenados oficiales
model = YOLO("yolo11n.pt")

# Iniciar entrenamiento
results = model.train(
    data="dataset_helmet.yaml",
    epochs=50,
    patience=15,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    device=0 if torch.cuda.is_available() else "cpu",
    project="runs/helmet_train",
    name="yolo11n_helmet",
    save=True
)

## 5. Validación y Métricas Formales

In [ ]:
# Evaluar sobre el subconjunto de validación
metrics = model.val(data="dataset_helmet.yaml", split="val")

print(f"Precision @ 0.5      : {metrics.box.mp:.4f}")
print(f"Recall @ 0.5         : {metrics.box.mr:.4f}")
print(f"mAP @ 0.5            : {metrics.box.map50:.4f}")
print(f"mAP @ 0.5:0.95       : {metrics.box.map:.4f}")

## 6. Exportación de Pesos (`best.pt` y ONNX)

In [ ]:
# Exportar a ONNX para acelerar inferencia en CPU de laptops
model.export(format="onnx", dynamic=False, simplify=True)

# Copiar los mejores pesos para descargar localmente a models/helmet_yolo11n.pt
!cp runs/helmet_train/yolo11n_helmet/weights/best.pt /content/helmet_yolo11n.pt
print("[✓] Pesos best.pt listos para descargar y ubicar en models/helmet_yolo11n.pt")